In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from joblib import Parallel, delayed
from datetime import datetime, timedelta

In [2]:
all_sample = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/all_sample.parquet')
factor1 = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/cne6_factor.parquet')
factor2 = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/other_factor.parquet')
factor3 = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/tech_factor.parquet')

In [94]:
sample = all_sample.query( 'trade_date >= "2023-01-01"') ## oot
#sample = all_sample.query( 'trade_date < "2023-01-01"') # train
sample = sample.merge(factor1, on=['ts_code', 'trade_date'], how='left').merge(
    factor2, on=['ts_code', 'trade_date'], how='left').merge(factor3, on=['ts_code', 'trade_date'], how='left')

In [ ]:
factor = sample.copy()
ind = pd.read_csv('C:/Users/User/OneDrive - CUHK-Shenzhen/data/industry.csv')
factor = factor.merge(ind[['ts_code', 'l1_name']], on='ts_code', how='left')
factor = pd.get_dummies(factor, columns=['l1_name'], drop_first=True, dtype=int)
#daily_corr = factor.groupby('trade_date').apply(lambda x: x[factor.columns[8:]].corrwith(x['Y_20d'])).reset_index()

In [97]:
def neutralize_date(group):
    X_raw = group[factor.columns[-30:].tolist() + ['lncap']].values
    X = np.column_stack((np.ones(X_raw.shape[0]), X_raw))
    # 预分配残差矩阵，默认全为 NaN
    resid_matrix = np.full((group.shape[0], 155), np.nan)
    # 2. 必须按列独立回归，处理各自的缺失值
    for j, col in enumerate(factor.columns[8:-30]):
        try:
            median = group[col].median()
            mad = (group[col] - median).abs().median()
            group[col] = group[col].clip(median - 5 * mad, median + 5 * mad)
            mean = group[col].mean()
            std = group[col].std()
            if std == 0:
                group[col] = np.nan  # 波动为0的因子无信息，置0
            else:
                group[col] = (group[col] - mean) / std
            
            Y_raw = group[col].values
            # 核心：只判断当前因子列的缺失值
            valid_idx = ~np.isnan(Y_raw) & np.all(~np.isnan(X), axis=1)
            X_valid = X[valid_idx]
            Y_valid = Y_raw[valid_idx]
            # 3. 底层 NumPy 最小二乘法求解
            beta, _, _, _ = np.linalg.lstsq(X_valid, Y_valid[:, np.newaxis])
            # 计算残差并展平为一维
            resid_valid = Y_valid - (X_valid @ beta).flatten()
            # 4. 只将有效行的残差填入对应位置，无效行保持 NaN
            resid_matrix[valid_idx, j] = resid_valid
        except:
            print(group.name, col)
    # 5. 映射回 DataFrame
    group[factor.columns[8:-30]] = resid_matrix
    return group
factor =  factor.groupby('trade_date').apply(neutralize_date).reset_index(level=0)
factor = factor.drop(columns=factor.columns[-30:].tolist() + ['lncap'])

C:\Users\User\AppData\Local\Temp\ipykernel_7900\1682278492.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  factor =  factor.groupby('trade_date').apply(neutralize_date).reset_index(level=0)


In [98]:
factor.to_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/sample_oot.parquet', index=False)

### 策略

In [99]:
from datatools import *

In [ ]:
class Portfolio:
    def __init__(self, initial_cash):
        self.cash = initial_cash
        self.positions = {}  # {stock_code: {'amount': int}}
        self.total_value = initial_cash
    def update_positions(self, current_prices):
        """根据最新价格更新总资产"""
        self.total_value = self.cash
        for stock, pos in self.positions.items():
            if stock in current_prices:
                self.total_value += pos['amount'] * current_prices[stock]
    def order_target_value(self, stock, target_value, current_price):
        """调整股票仓位至目标市值"""
        if current_price <= 0: return
        current_value = self.positions.get(stock, {}).get('amount', 0) * current_price
        diff = target_value - current_value
        if diff > 0: # 买入
            shares = int(diff / current_price / 100) * 100 # 按手交易
            if shares > 0:
                cost = shares * current_price  * (1+0.0013) # 简单手续费
                if cost < self.cash:
                    self.cash -= cost
                    self.positions[stock] = {'amount': self.positions.get(stock, {}).get('amount', 0) + shares}
        elif diff < 0: # 卖出
            shares = int(abs(diff) / current_price / 100) * 100
            if shares > 0:
                hold = self.positions.get(stock, {}).get('amount', 0)
                shares = min(shares, hold)
                if shares > 0:
                    revenue = shares * current_price  * (1 - 0.0013)
                    self.cash += revenue
                    self.positions[stock]['amount'] -= shares
                    if self.positions[stock]['amount'] == 0:
                        del self.positions[stock]
    def order_target(self, stock, target_amount, current_price):
        """调整股票仓位至目标股数"""
        if current_price <= 0: return
        current_amount = self.positions.get(stock, {}).get('amount', 0)
        diff = target_amount - current_amount
        if diff > 0:
            cost = diff * current_price * (1+ 0.0013)
            if cost < self.cash:
                self.cash -= cost
                self.positions[stock] = {'amount': current_amount + diff}
        elif diff < 0:
            revenue = abs(diff) * current_price * (1 - 0.0013)
            self.cash += revenue
            self.positions[stock]['amount'] += diff
            if self.positions[stock]['amount'] == 0:
                del self.positions[stock]

class Context:
    """模拟聚宽的 context 对象"""
    def __init__(self, start_date, initial_cash=1000000):
        self.current_dt = start_date
        self.previous_date = start_date - timedelta(days=1)
        self.portfolio = Portfolio(initial_cash)


# ==========================================
# 4. 本地回测主程序入口
# ==========================================
start_date = datetime(2021, 1, 1)
end_date = datetime(2024, 1, 1)
initial_cash = 100000
context = Context(start_date, initial_cash)
N = 18
M = 1100
init = True
stock_num = 10
security = ['000300.XSHG'] # 本地运行时此处可替换为实际代码或名称
days = 0
ans = []
ans_rightdev = []
# 计算2015年1月5日至回测开始日期的RSRS斜率指标
prices = get_price(security, '2015-01-01', context.previous_date, '1d', ['high', 'low'])
highs = prices.high
lows = prices.low
for i in range(len(highs))[N:]:
    data_high = highs.iloc[i-N+1:i+1]
    data_low = lows.iloc[i-N+1:i+1]
    X = sm.add_constant(data_low)
    model = sm.OLS(data_high, X)
    results = model.fit()
    ans.append(results.params[1])
    ans_rightdev.append(results.rsquared)

# 主循环：按日遍历
for current_date in get_trade_cal(start_date=start_date, end_date=end_date):
    context.current_dt = current_date
    context.previous_date = current_date - timedelta(days=1)
    current_price_df = get_price(security, current_date, current_date, '1d', ['close'])
    # 执行回测流程
    beta = 0
    r2 = 0
    if init:
        init = False
    else:
        prices = get_price(security, N, '1d', ['high', 'low'])
        highs = prices.high
        lows = prices.low
        X = sm.add_constant(lows)
        model = sm.OLS(highs, X)
        beta = model.fit().params[1]
        ans.append(beta)
        r2 = model.fit().rsquared
        ans_rightdev.append(r2)
    # 计算标准化的RSRS指标
    section = ans[-M:]
    mu = np.mean(section)
    sigma = np.std(section)
    zscore = (section[-1] - mu) / sigma  
    zscore_rightdev = zscore * beta * r2
    if zscore_rightdev > 0.7:
        df = get_basic(context.current_dt)
        df = df[(df['roe'] > 0) & (df['pb_ratio'] > 0)].sort_values('pb_ratio')
        df.index = df['code'].values
        df['1/roe'] = 1 / df['roe']
        df['point'] = df[['pb_ratio', '1/roe']].rank().T.apply(sum)
        df = df.sort_values('point')[:stock_num]
        pool = df.index
        print(f'总共选出 {len(pool)} 只股票')
        if len(pool) == 0:
            continue
        cash = context.portfolio.total_value / len(pool)
        hold_stock = list(context.portfolio.positions.keys())
        # 获取股票当前价格（此处需实现获取一篮子股票价格的逻辑）
        # 简化处理：假设所有股票价格为10元，实际应用需替换为真实行情获取
        stock_prices = {s: 10.0 for s in pool} 
        for s in hold_stock:
            if s not in pool:
                cp = stock_prices.get(s, 10.0) # 卖出时也需要价格
                context.portfolio.order_target(s, 0, cp)
        for s in pool:
            cp = stock_prices.get(s, 10.0)
            context.portfolio.order_target_value(s, cash, cp)
    elif (zscore_rightdev < -0.7) and (len(context.portfolio.positions.keys()) > 0):
        # 获取当前价格用于平仓计算
        current_prices = get_price(security, context.current_dt, context.current_dt, '1d', ['close'])
        for s in list(context.portfolio.positions.keys()):
            # 假设持仓股票代码可以获取价格，这里简化使用指数价格替代，实际需替换为个股真实价格
            cp = current_prices['close'].iloc[0] if not current_prices.empty else 1.0 
            context.portfolio.order_target(s, 0, cp)
    # 每日结算后更新一次总资产
    context.portfolio.update_positions({security: current_price_df['close'].iloc[0] if not current_price_df.empty else 0})